## **Profile a simple CNN trained on a synthetic data.**
All code is in single cell, just select the cell and run with nsight profiler (*green run button* from the toolbar). 
If you like to profile a portion of code, split the code into multiple cells and profile the cell you like to investigate. <br/>
As an alternate you could also add **nvtx** annotations for the code snippet you want to dig deeper. Example with **nvtx** is in the following notebook -

    debug-llm-bottlenecks-with-nvidia-nsight/lab-01/01-simple-cnn-with-nvtx/simple-cnn-with-nvtx-baseline.ipynb

**Pls Note:** More details on the bottlenecks that can be observed as part of the below code snippet are documented at the end of this notebook.

In [ ]:
!pip install matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 256 * 4 * 4)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Create synthetic data
batch_size = 64
data = torch.randn(5000, 3, 32, 32)
targets = torch.randint(0, 10, (5000,))

# Visualize synthetic data images
def visualize_image(img, target):
    image = img.permute(1, 2, 0)
    image_norm = (image - image.min()) / (image.max() - image.min())
    plt.figure(figsize=(4, 4))
    plt.imshow(image_norm)
    plt.title(f"Sample Image - Label: {target.item()}")
    plt.axis('off')
    plt.show()
# display images to see the generated data
visualize_image (data[0], targets[0])
visualize_image (data[1], targets[1])

dataset = TensorDataset(data, targets)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


# Training loop
def train_step():
    model.train()
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")
        # if batch_idx >= 10:  # Profile just a few batches
        #     break

# Run training
torch.cuda.synchronize()
train_step()
torch.cuda.synchronize()

## GPU Compute Bottlenecks

**Convolution Operations**
• conv1, conv2, conv3: Most computationally expensive
• Kernel launches and CUDA core utilization
• Memory access patterns for convolution windows

**Matrix Multiplications**
• fc1: 4096 → 512 linear layer (large matrix multiply)
• fc2: 512 → 10 final classification layer

**Activation Functions**
• ReLU operations after each convolution
• Element-wise operations across large tensors

## Memory Bottlenecks

**Data Transfers**
data, target = data.to(device), target.to(device)  # CPU → GPU transfer


**Memory Allocation**
• Intermediate tensors during forward pass
• Gradient storage during backward pass
• Activation maps: 64×16×16, 128×8×8, 256×4×4

**Memory Bandwidth**
• Reading weights and activations
• Writing gradients and updated parameters

## Synchronization Bottlenecks

**CPU-GPU Synchronization**
    loss.item()  # Forces GPU → CPU transfer
    print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")  # Blocks execution


**DataLoader Bottlenecks**
• Single-threaded data loading (no num_workers)
• CPU data preparation while GPU is idle

## Gradient Computation Bottlenecks

**Backward Pass**
       loss.backward()  # Computes gradients for all parameters

• Backpropagation through all layers
• Gradient accumulation and storage

**Optimizer Updates**
        optimizer.step()  # Updates all model parameters


## Specific Hotspots Nsight Will Show

1. Conv2d kernels: Highest GPU utilization
2. Data movement: Host-to-device transfers
3. Memory allocation: Dynamic tensor creation
4. Gradient computation: Backward pass operations
5. Parameter updates: Optimizer step operations

## **Quick Optimizations**

### Add these to reduce bottlenecks:
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, 
                       num_workers=4, pin_memory=True)  # Faster data loading

### Remove synchronous operations
 print(f"Loss: {loss.item():.4f}")  # Comment out during profiling

### Use mixed precision
  model = model.half()  # FP16 for faster computation
<br/>

## **Refer below notebook for referrence implementation of these optimizations**
    debug-llm-bottlenecks-with-nvidia-nsight/lab-01/01-simple-cnn/simple-cnn-optimized.ipynb

## **Sample Profile Reports**

![](../..static-assets/lab-01/simple-cnn-baseline.png)

![](../../static-assets/lab-01/simple-cnn-baseline.png)

![](../../static-assets/lab-01/simple-cnn-baseline-memops.png)

![](../../static-assets/lab-01/simple-cnn-baseline-DtoH.png)

![](../../static-assets/lab-01/simple-cnn-baseline-gpu.png)